In [ ]:
!python --version

### Get OWL files

https://reactome.org/download/current/biopax/{pathway_id}_level3.owl

In [ ]:
import sys
from pathlib import Path

from pybiopax import model_from_owl_file
from collections import Counter

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import *
from libs.dashcyto_lib import DASH_CYTO
from libs.reactome_lib import Reactome


In [ ]:
rea = Reactome(ROOT0=ROOT0)

rea.root_reactome, rea.root_owl

In [ ]:
dfr = rea.open_reactome()
print(len(dfr))
dfr.head(3)

### pybiopax

uv pip install pybiopax  
uv pip install ipywidgets  

In [ ]:
hsa_model = rea.open_human_owl(force=False)

print(type(hsa_model))
print(len(hsa_model.objects))


In [ ]:
class_counts = Counter(type(obj).__name__ for obj in hsa_model.objects.values())

i=0
for cls, n in class_counts.most_common(20):
    print(cls, n)
    i+=1
    if i==10: break

In [ ]:
pathways, pathway_index = rea.get_pathways_from_model()
print("Number of pathways:", len(pathways))
print("Number of pathway IDs:", len(pathway_index))


In [ ]:
dfo = rea.pathways_to_df(pathways)
dfo.head(12)

In [ ]:
for p in pathways[:3]:
    print(p.uid, getattr(p, "display_name", None), getattr(p, "pid", None)) 

In [ ]:
i=0
for key, pathw in pathway_index.items():
    print(f"{key:15} {pathw}  {pathw.uid} {pathw.display_name}")
    i+=1
    if i==3: break

In [ ]:
pid = "R-HSA-164843"

p = pathway_index.get(pid)

if p is None:
    print("Not found")
else:
    print("Found:", p.display_name)

### Recursively collect all referenced objects

In [ ]:
i=0
p = pathways[i]
print(f"Pathway {i}: {p.uid} - {p.display_name}")


sub_objects = rea.collect_referenced_objects(p, include_inverse=False)

print("Objects in pathway subset:", len(sub_objects))

In [ ]:
uid, pathway_id, pathway = rea.get_pathway_names(p, dfo)
uid, pathway_id, pathway

In [ ]:
verbose=True
force=False

ret = rea.save_owl_model(pathway_id, pathway, sub_objects=sub_objects, force=force, verbose=verbose)
ret

### Next check is the OWL export function available in your installed pybiopax:

In [ ]:
import pybiopax

print([x for x in dir(pybiopax) if "owl" in x.lower()])

In [ ]:
fname_owl = rea.fname_owl%(pathway_id)

filename = rea.root_owl / fname_owl
test_model = pybiopax.model_from_owl_file(filename)

print(type(test_model))
print(len(test_model.objects))

### DASH_CYTO

In [ ]:
dcy = DASH_CYTO(ROOT0=ROOT0)

rdf = dcy.read_owl(pathway_id, pathway)
height = "1100px"
width = "100%"
marginTop="20px"

dcy.run_app(height=height, width=width, marginTop=marginTop)

### Read and Plot any pathway

In [ ]:
dcy = DASH_CYTO(ROOT0=ROOT0)

pathway = 'Chaperone Mediated Autophagy'
pathway_id = 'R-HSA-9613829'

rdf = dcy.read_owl(pathway_id, pathway)
height = "1100px"
width = "100%"
marginTop="20px"

dcy.run_app(height=height, width=width, marginTop=marginTop)